In [ ]:
from typing import Any, SupportsFloat
import pygame
import random
import numpy as np
import time
import keyboard
import copy
import numpy
import gymnasium as gym
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecEnv
from gymnasium.wrappers import TimeLimit
import cv2
from gymnasium.envs.registration import register


class SnakeEnv:

    def __init__(self, height, width) -> None:
        self.height = height
        self.width = width
        self.observationSpace = [
            [0 for _ in range(self.width)] for _ in range(self.height)]
        pass

    def getValue(self, y, x):
        return self.observationSpace[y][x]

    def setValue(self, y, x, value):
        if self.observationSpace[y][x] != value:
            self.observationSpace[y][x] = value

    def setNull(self, y, x):
        self.observationSpace[y][x] = 0

    def print(self):
        print()
        for y in range(len(self.observationSpace)):
            print(self.observationSpace[y])

    def getDesignationValue(self, value):
        result = []
        for y in range(len(self.observationSpace)):
            for x in range(len(self.observationSpace[y])):
                if self.observationSpace[y][x] == value:
                    result.append((y, x))
        return result

    def getEnvironmentalData(self):
        return self.observationSpace


import random
from collections import deque

class Wall:
    def __init__(self, snakeEnv: SnakeEnv, num_interior_walls: int = 0) -> None:
        self.value = 1
        self.snakeEnv = snakeEnv
        self.walls = []
        self.num_interior_walls = num_interior_walls
        self.generate()

    def generate(self):
        snakeEnv = self.snakeEnv
        # Always place border walls
        for y in range(snakeEnv.height):
            for x in range(snakeEnv.width):
                if x == 0 or x == snakeEnv.width - 1 or \
                        y == 0 or y == snakeEnv.height - 1:
                    snakeEnv.setValue(y, x, self.value)
                    self.walls.append((y, x))

        # Try to place interior walls randomly
        placed = 0
        attempts = 0
        max_attempts = self.num_interior_walls * 20  # avoid infinite loops

        while placed < self.num_interior_walls and attempts < max_attempts:
            attempts += 1
            # Pick a random interior cell
            y = random.randint(1, snakeEnv.height - 2)
            x = random.randint(1, snakeEnv.width - 2)

            if self.isWall(y, x):
                continue

            # Tentatively add wall and check safety
            self.walls.append((y, x))
            if self._is_safe():
                snakeEnv.setValue(y, x, self.value)
                placed += 1
            else:
                # Reject — would create a deadlock cell
                self.walls.pop()

    def _is_safe(self) -> bool:
        """
        Check two conditions:
        1. All non-wall cells are still connected (no isolated regions).
        2. No non-wall cell has fewer than 2 non-wall neighbours
           (a cell with only 1 free neighbour is a dead-end that
           prevents a Hamiltonian path on the interior grid).
        """
        snakeEnv = self.snakeEnv
        wall_set = set(self.walls)

        free_cells = [
            (y, x)
            for y in range(snakeEnv.height)
            for x in range(snakeEnv.width)
            if (y, x) not in wall_set
        ]

        if not free_cells:
            return False

        # connectivity via BFS
        visited = set()
        queue = deque([free_cells[0]])
        visited.add(free_cells[0])

        while queue:
            cy, cx = queue.popleft()
            for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]:
                nb = (cy + dy, cx + dx)
                if nb not in visited and nb not in wall_set \
                        and 0 <= nb[0] < snakeEnv.height \
                        and 0 <= nb[1] < snakeEnv.width:
                    visited.add(nb)
                    queue.append(nb)

        if len(visited) != len(free_cells):
            return False  # disconnected region

        #no dead-end cells
        for (cy, cx) in free_cells:
            free_neighbours = sum(
                1 for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]
                if (cy + dy, cx + dx) not in wall_set
                and 0 <= cy + dy < snakeEnv.height
                and 0 <= cx + dx < snakeEnv.width
            )
            if free_neighbours < 2:
                return False  # dead-end cell -> Hamiltonian path impossible

        return True

    def isWall(self, y, x):
        return (y, x) in set(self.walls)



    def reset(self):
        snakeEnv = self.snakeEnv

        # Clear only interior walls from the grid
        for (y, x) in self.walls:
            if not self._is_border(y, x):
                snakeEnv.setValue(y, x, 0)

        # Keep only border walls
        self.walls = [(y, x) for (y, x) in self.walls if self._is_border(y, x)]

        # Re-place border walls on the fresh grid
        for (y, x) in self.walls:
            snakeEnv.setValue(y, x, self.value)

        # Re-run interior placement
        placed = 0
        attempts = 0
        max_attempts = self.num_interior_walls * 20

        while placed < self.num_interior_walls and attempts < max_attempts:
            attempts += 1
            y = random.randint(1, snakeEnv.height - 2)
            x = random.randint(1, snakeEnv.width - 2)

            if self.isWall(y, x):
                continue

            self.walls.append((y, x))
            if self._is_safe():
                snakeEnv.setValue(y, x, self.value)
                placed += 1
            else:
                self.walls.pop()

    def _is_border(self, y, x) -> bool:
        snakeEnv = self.snakeEnv
        return x == 0 or x == snakeEnv.width - 1 or \
            y == 0 or y == snakeEnv.height - 1


class Food:

    def __init__(self, snakeEnv: SnakeEnv, seed=None) -> None:
        self.value = 2
        self.snakeEnv = snakeEnv
        if seed == None:
            seed = int(time.time())
        random.seed(seed)
        self.foodCoordinates = ()
        self.generate()
        pass

    def generate(self, remove=False):
        if remove and len(self.foodCoordinates) == 2:
            y, x = self.foodCoordinates
            self.snakeEnv.setNull(y, x)
        emptyPosition = self.snakeEnv.getDesignationValue(0)
        lenEmptyPosition = len(emptyPosition)
        if lenEmptyPosition <= 0:
            return True
        if lenEmptyPosition == 1:
            y, x = emptyPosition[0]
        else:
            ri = random.randint(0, len(emptyPosition) - 1)
            y, x = emptyPosition[ri]
        self.foodCoordinates = (y, x)
        self.snakeEnv.setValue(y, x, self.value)
        return False

    def getFoodCoordinates(self):
        return self.foodCoordinates

    def isFood(self, y, x):
        vy, vx = self.foodCoordinates
        return vy == y and vx == x

    def rendering(self):
        if len(self.foodCoordinates) == 2:
            y, x = self.foodCoordinates
            self.snakeEnv.setValue(y, x, self.value)


class SnakeBody:

    UP = 1
    DOWN = 2
    LEFT = 3
    RIGHT = 4

    def __init__(self, snakeEnv: SnakeEnv) -> None:
        self.snakeEnv = snakeEnv
        self.head_value = 3
        self.body_value = 4
        # Length
        self.len = 1
        # Direction of movement
        self.movingDirection = 1
        self.body = [(int(snakeEnv.height / 2), int(snakeEnv.width / 2))]
        self.end = ()
        self.generate()
        pass

    def getBody(self):
        return self.body

    def getHead(self):
        return self.body[0]

    def length(self):
        return len(self.body)

    def setAction(self, action):
        if action == 0:
            return
        if (action % 2 == 0 and action - 1 == self.movingDirection) or \
                (action % 1 == 0 and action + 1 == self.movingDirection):
            return
        self.movingDirection = action

    def lengthen(self):
        if len(self.end) == 2:
            y, x = self.end
            self.body.append((y, x))
            self.snakeEnv.setValue(y, x, self.body_value)
            self.end = ()

    def move(self):
        y, x = self.body[0]
        if self.movingDirection == SnakeBody.UP:
            y -= 1
        elif self.movingDirection == SnakeBody.DOWN:
            y += 1
        elif self.movingDirection == SnakeBody.LEFT:
            x -= 1
        elif self.movingDirection == SnakeBody.RIGHT:
            x += 1
        else:
            raise RuntimeError(f"Exception move : {self.movingDirection}")
        self.body.insert(0, (y, x))
        ey, ex = self.body.pop()
        self.end = (ey, ex)
        self.snakeEnv.setNull(ey, ex)
        self.generate()
        return (y, x)

    def headBodyStack(self):
        hy, hx = self.body[0]
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            if hy == y and hx == x:
                return True
        return False

    def isBody(self, hy, hx):
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            if hy == y and hx == x:
                return True
        return False

    def generate(self):
        y, x = self.body[0]
        self.snakeEnv.setValue(y, x, self.head_value)
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            self.snakeEnv.setValue(y, x, self.body_value)


class TcsV2Env(gym.Env):

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, **kwargs) -> None:
        self.image_size = kwargs.get("image_size", 10)
        self.show = False
        self.height = kwargs.get("height", 50)
        self.width = kwargs.get("width", 50)
        self.windowcolor = [(0, 0, 0), (255, 255, 255), (255, 0, 0),
                            (0, 0, 255), None]
        self.WINDOW_WIDTH = self.width * self.image_size
        self.WINDOW_HEIGHT = self.height * self.image_size
        self.WINDOW_BLACK = (0, 0, 0)
        self.obs_type = kwargs.get("obs_type", "rgb")
        self.outputType = kwargs.get("outputType", "Discrete")
        if self.outputType == "Discrete":
            self.action_space = gym.spaces.Discrete(5)
        elif self.outputType == "Box":
            self.action_space = gym.spaces.Box(
                low=0, high=4, shape=(1,), dtype=np.uint8
            )
        if self.obs_type == "rgb":
            self.observation_space = gym.spaces.Box(
                low=0, high=255, shape=(self.height * self.image_size, self.width * self.image_size, 3), dtype=np.uint8)
        if self.obs_type == "data":
            self.observation_space = gym.spaces.Box(
                low=0, high=255, shape=(8,), dtype=np.float32)
        self.render_mode = kwargs.get("render_mode", None)
        self.mp4 = kwargs.get("mp4", False)
        self.mp4fps = kwargs.get("mp4fps", 30)
        assert self.render_mode is None or self.render_mode in self.metadata["render_modes"]
        super().__init__()

    def get_action_meanings(self):
        return ["NOOP", "UP", "DOWN", "LEFT", "RIGHT"]

    def step(self, action: Any) -> tuple[Any, SupportsFloat, bool, bool, dict[str, Any]]:
        if isinstance(action, np.ndarray):
            action = action[0]
            action = round(action)
        info = {}
        reward = 0
        self.snakebody.setAction(action)
        y, x = self.snakebody.move()
        done = False
        if self.snakebody.isBody(y, x):
            done = True
            info["message"] = "Come into contact with the body"
        if self.wall.isWall(y, x):
            done = True
            info["message"] = "hit the wall"
        if self.food.isFood(y, x):
            self.snakebody.lengthen()
            if self.food.generate():
                done = True
                vn_step = 1000 - self.num_step
                if vn_step < 0:
                    vn_step = 0
                reward += 100 * (vn_step / 1000)
                info["message"] = "Damn, the screen is completely covered with it"
                if self.show:
                    time.sleep(3)
            reward += 1
            # self.distanceFood = 20
        # else:
        #     fy, fx = self.food.getFoodCoordinates()
        #     jy = abs(fy - y)
        #     jx = abs(fx - x)
        #     distance = jy + jx
        #     if distance < self.distanceFood:
        #         self.distanceFood = distance
        #         reward += 1 - ((1 / 20) * distance)
        if reward < 0:
            reward = 0


        self.score += reward
        observation = self.environmentalData()

        if self.mp4 and self.obs_type != "rgb":
            self.render()

        self.num_step += 1
        return observation, reward, done, False, info

    def reset(self, *, seed: int | None = None, options: dict[str, Any] | None = None) -> tuple[Any, dict[str, Any]]:
        self.snakeEnv = SnakeEnv(self.height, self.width)

        if hasattr(self, 'wall'):
            self.wall.snakeEnv = self.snakeEnv  # point wall to the fresh grid
            self.wall.reset()
        else:
            self.wall = Wall(self.snakeEnv)

        self.snakebody = SnakeBody(self.snakeEnv)
        self.food = Food(self.snakeEnv, seed=seed)
        observation = self.environmentalData()
        self.score = 0
        self.num_step = 0
        # self.distanceFood = 20
        return observation, {}


    # Changed observations, added danger observations
    # new observation space 225 * 16 * 4 * 36 = 518_400 rows
    def environmentalData(self): # GUIDING, not STATING

        if self.obs_type == "data":
            hy, hx = self.snakebody.getHead()
            fy, fx = self.food.getFoodCoordinates()

            # Relative food position (more signal)
            food_dy = fy - hy
            food_dx = fx - hx

            # Danger flags in each direction
            danger_up = int(self.wall.isWall(hy-1, hx) or self.snakebody.isBody(hy-1, hx))
            danger_down = int(self.wall.isWall(hy+1, hx) or self.snakebody.isBody(hy+1, hx))
            danger_left = int(self.wall.isWall(hy, hx-1) or self.snakebody.isBody(hy, hx-1))
            danger_right = int(self.wall.isWall(hy, hx+1) or self.snakebody.isBody(hy, hx+1))

            # Current direction (1–4)
            direction = self.snakebody.movingDirection

            # Snake length (normalised)
            length = self.snakebody.length()

            return np.array([food_dy, food_dx,
                 danger_up, danger_down, danger_left, danger_right,
                 direction,
                 length], dtype=np.float32)

        elif self.obs_type == "rgb":
            observation = self.render()
            return observation

        return
    def render(self):
        if (self.render_mode == "human") and ("window" not in self.__dict__ or self.window == None):
            pygame.init()
            self.window = pygame.display.set_mode(
                (self.width * 10, self.height * 10))
            pygame.display.set_caption("tcs")
            self.window.fill(self.WINDOW_BLACK)
            self.show = True
        if "canvas" not in self.__dict__ or self.canvas == None:
            self.canvas = pygame.Surface(
                (self.WINDOW_WIDTH, self.WINDOW_HEIGHT))
        if self.show and ("canvasN" not in self.__dict__ or self.canvasN == None):
            self.canvasN = pygame.Surface(
                (self.width * 10, self.height * 10)
            )
        self.canvas.fill(self.WINDOW_BLACK)
        if self.show:
            self.canvasN.fill(self.WINDOW_BLACK)

        observation = self.snakeEnv.getEnvironmentalData()
        for y in range(len(observation)):
            for x in range(len(observation[y])):
                value = observation[y][x]
                if value == 0 or self.windowcolor[value] == None:
                    continue
                pygame.draw.rect(
                    self.canvas, self.windowcolor[value], (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
                if self.show:
                    pygame.draw.rect(
                        self.canvasN, self.windowcolor[value], (
                            x * 10, y * 10, 10, 10)
                    )
        body = self.snakebody.getBody()
        bodylen = len(body)
        if bodylen > 1:
            colosNm = round(105 / (bodylen - 1))
            dcolosNm = 150 + colosNm
            for i in range(bodylen - 2, 0, -1):
                y, x = body[i]
                vvColos = (0, dcolosNm, 0)
                pygame.draw.rect(
                    self.canvas, vvColos, (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
                if self.show:
                    pygame.draw.rect(
                        self.canvasN, vvColos, (
                            x * 10, y * 10, 10, 10)
                    )
                if dcolosNm >= 255:
                    continue
                dcolosNm += colosNm
                if dcolosNm > 255:
                    dcolosNm = 255
            y, x = body[-1]
            vbcolos = (232, 255, 1)
            pygame.draw.rect(
                self.canvas, vbcolos, (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
            if self.show:
                pygame.draw.rect(
                    self.canvasN, vbcolos, (
                        x * 10, y * 10, 10, 10)
                )

        result = np.transpose(
            np.array(pygame.surfarray.pixels3d(self.canvas)), axes=(1, 0, 2)
        )
        if self.render_mode == "human":
            show_image = np.transpose(
                np.array(pygame.surfarray.pixels3d(self.canvasN)), axes=(1, 0, 2)
            )
            self.window.blit(self.canvasN, self.canvasN.get_rect())
            pygame.event.pump()
            pygame.display.update()
            # pygame.display.flip()
            # event = pygame.event.poll()
            # # pygame.display.update()
            # if event.type == pygame.QUIT:
            #     pygame.quit()
            #     exit()
            if self.mp4:
                if "outMp4" not in self.__dict__ or self.outMp4 == None:
                    height, width, _ = show_image.shape
                    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                    self.outMp4 = cv2.VideoWriter(
                        "1.mp4", fourcc, self.mp4fps, (width, height))
                self.outMp4.write(show_image)

        return result

    def close(self):
        del self.snakebody
        del self.wall
        del self.food
        del self.snakeEnv
        del self.score
        if self.mp4 and hasattr(self, "outMp4") and self.outMp4 is not None:
            self.outMp4.release()
        return super().close()


class ConstructEnv:

    def __init__(self) -> None:
        gym.register(id="tcs-v2", entry_point=TcsV2Env, max_episode_steps=2000,
                     kwargs={})
        pass

    def getDummyVecEnvs(self, num=32, **kwargs) -> DummyVecEnv:
        return DummyVecEnv([lambda: self.getEnv(**kwargs) for _ in range(num)])

    def getSudoVecEnvs(self, num=6, **kwargs) -> SubprocVecEnv:
        return SubprocVecEnv([lambda: self.getEnv(**kwargs) for _ in range(num)])

    def getEnv(self, **kwargs) -> TimeLimit:
        return TimeLimit(gym.make("tcs-v2", **kwargs), max_episode_steps=5000)




register(
    id="tcs-v2", entry_point=TcsV2Env, max_episode_steps=2000,
                     kwargs={}
)

c:\Users\adjd119\.conda\envs\DRL_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment tcs-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [ ]:
import gymnasium as gym    

#no penalty - snake learns to survive
#too large a penalty - snake learns to die quickly
class RewardShapingWrapper(gym.Wrapper):
    def __init__(self, env, step_penalty=-0.01, hunger_cap=36):
        super().__init__(env)
        self.step_penalty = step_penalty
        self.hunger_cap = hunger_cap
        self.steps_since_food = 0

    def reset(self, **kwargs):
        self.steps_since_food = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)

        if reward >= 1:  # food eaten (your env gives +1 per food)
            self.steps_since_food = 0
        else:
            self.steps_since_food += 1

        if self.steps_since_food >= self.hunger_cap:
            terminated = True
            info["message"] = "Starved"
            self.steps_since_food = 0

        reward += self.step_penalty
        return obs, reward, terminated, truncated, info
    
    


In [ ]:
import wandb
# from dotenv import load_dotenv
# import os
#
# load_dotenv()
    # wandb_key = os.getenv("WANDB_KEY")
try:
    wandb.login(key="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")
    api = wandb.Api()
    viewer = api.viewer
    print(f"Logged in as: {viewer.username}")
except wandb.errors.AuthenticationError:
    print("Not logged in")

Eval pipeline to generate video and eval results

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import wandb
from torch.distributions import Categorical, Normal
from stable_baselines3.common.env_util import make_vec_env


class A2CNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, is_continuous=False):
        super().__init__()
        self.is_continuous = is_continuous

        self.critic = nn.Sequential(
            nn.Linear(state_dim, 64), nn.Tanh(),
            nn.Linear(64, 64),        nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, action_dim)
        )


    def get_action_and_value(self, x, action=None):
        value = self.critic(x)
        if not self.is_continuous:
            logits = self.actor(x)
            dist   = Categorical(logits=logits) # dist is variable probs in Lab7 tutorial code
            if action is None:
                action = dist.sample()
            return action, dist.log_prob(action), dist.entropy(), value

        else:
            mu  = self.actor(x)
            std = torch.exp(self.log_std)
            dist = Normal(mu, std)
            if action is None:
                action = dist.sample()
            return action, dist.log_prob(action).sum(1), dist.entropy().sum(1), value

    def get_value(self, x):
        return self.critic(x)

class RolloutBuffer:
    def __init__(self, n_steps, n_envs, state_dim, device):
        self.n_steps   = n_steps
        self.n_envs    = n_envs
        self.state_dim = state_dim
        self.device    = device
        self.reset()

    def reset(self):
        self.observations = torch.zeros(self.n_steps, self.n_envs, self.state_dim).to(self.device)
        self.actions      = torch.zeros(self.n_steps, self.n_envs,               ).to(self.device)
        self.log_probs    = torch.zeros(self.n_steps, self.n_envs,               ).to(self.device)
        self.rewards      = torch.zeros(self.n_steps, self.n_envs,               ).to(self.device)
        self.dones        = torch.zeros(self.n_steps, self.n_envs,               ).to(self.device)
        self.values       = torch.zeros(self.n_steps, self.n_envs,               ).to(self.device)
        self.ptr          = 0  # current write position

    def add(self, obs, action, log_prob, reward, done, value):
        self.observations[self.ptr] = obs
        self.actions     [self.ptr] = action
        self.log_probs   [self.ptr] = log_prob
        self.rewards     [self.ptr] = reward
        self.dones       [self.ptr] = done
        self.values      [self.ptr] = value
        self.ptr += 1

    def is_full(self):
        return self.ptr >= self.n_steps

    def compute_returns_and_advantages(self, last_value, gamma, gae_lambda):
        advantages = torch.zeros_like(self.rewards)
        last_gae = 0.0

        for t in reversed(range(self.n_steps)):
          if t == self.n_steps - 1:
            next_value = last_value
          else:
            next_value = self.values[t + 1]

          next_non_terminal = self.dones[t]

            # TD error
        delta = self.rewards[t] + gamma * next_value * next_non_terminal - self.values[t]
        last_gae = delta + gamma * gae_lambda * next_non_terminal * last_gae
        advantages[t] = last_gae

        returns = advantages + self.values
        return returns, advantages


class CustomA2C:
    def __init__(
        self,
        env,
        state_dim,
        action_dim,
        learning_rate = 7e-4,
        gamma         = 0.99,
        gae_lambda    = 1.0,
        ent_coef      = 0.01,   # entropy bonus weight — exploration mechanism
        vf_coef       = 0.5,    # critic loss weight
        max_grad_norm = 0.5,    # gradient clipping
        n_steps       = 5,      # rollout length per env before each update
        device        = "cpu",
    ):
        self.env           = env
        self.gamma         = gamma
        self.gae_lambda    = gae_lambda
        self.ent_coef      = ent_coef
        self.vf_coef       = vf_coef
        self.max_grad_norm = max_grad_norm
        self.n_steps       = n_steps
        self.n_envs        = env.num_envs
        self.device        = torch.device(device)

        self.policy = A2CNetwork(state_dim, action_dim).to(self.device)
        self.optimizer = torch.optim.RMSprop(
            self.policy.parameters(),
            lr=learning_rate,
            eps=1e-5,
            alpha=0.99,
        )

        self.buffer = RolloutBuffer(n_steps, self.n_envs, state_dim, self.device)


    def _train_step(self):

        obs = self.buffer.observations.view(-1, self.buffer.state_dim)
        actions = self.buffer.actions.view(-1).long()

        _, log_probs, entropy, values = self.policy.get_action_and_value(obs, actions)


        with torch.no_grad():
            last_obs = torch.tensor(self._last_obs, dtype=torch.float32).to(self.device)
            last_value = self.policy.get_value(last_obs)

        returns, advantages = self.buffer.compute_returns_and_advantages(
            last_value.squeeze(), self.gamma, self.gae_lambda
        )

        returns = returns.view(-1)
        advantages = advantages.view(-1)

        # Normalise advantages — reduces variance without changing the policy gradient direction
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        values = values.squeeze()

        policy_loss = -(advantages.detach() * log_probs).mean()
        value_loss = nn.functional.mse_loss(values, returns.detach())
        entropy_loss = -entropy.mean()

        loss = policy_loss + self.vf_coef * value_loss + self.ent_coef * entropy_loss

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
        self.optimizer.step()

        return {
            "loss/policy": policy_loss.item(),
            "loss/value":  value_loss.item(),
            "loss/entropy": entropy_loss.item(),
            "loss/total":   loss.item(),
        }


    def learn(self, total_timesteps):

        obs = self.env.reset()
        self._last_obs = obs.astype(np.float32)

        timestep = 0
        episode_count = 0

        while timestep < total_timesteps:
            self.buffer.reset()
            self.policy.eval()

            # Rollouts
            for _ in range(self.n_steps):
                with torch.no_grad():
                    obs_tensor = torch.tensor(self._last_obs, dtype=torch.float32).to(self.device)
                    action, log_prob, _, value = self.policy.get_action_and_value(obs_tensor)

                action_np = action.cpu().numpy()
                next_obs, reward, done, _ = self.env.step(action_np)

                self.buffer.add(
                    obs_tensor,
                    action,
                    log_prob,
                    torch.tensor(reward, dtype=torch.float32).to(self.device),
                    torch.tensor(done, dtype=torch.float32).to(self.device),
                    value.squeeze(),
                )

                self._last_obs = next_obs.astype(np.float32)
                timestep += self.n_envs
                episode_count += int(done.sum())

            # update nets
            self.policy.train()
            losses = self._train_step()


            if wandb.run is not None:
                wandb.log({"timestep": timestep, **losses})

                if timestep % 20000 == 0:


                    mean_reward, std_reward = evaluate(self, n_episodes=10)

                    wandb.log({
                        "timestep": timestep,
                        "mean_r": mean_reward,
                        "std_r": std_reward,
                    })
        return self


    def predict(self, obs, deterministic=True):

        self.policy.eval()
        with torch.no_grad():
            obs_tensor = torch.tensor(np.array(obs), dtype=torch.float32).to(self.device)
            if deterministic:
                logits = self.policy.actor(obs_tensor)
                action = logits.argmax(dim=-1)
            else:
                action, _, _, _ = self.policy.get_action_and_value(obs_tensor)
        return action.cpu().numpy(), None




def evaluate(model, n_episodes=10):
    vec_env_evaluate = make_vec_env(
        "tcs-v2",
         #add more envs until cpu cannot handle it
        #cpu is the main bottleneck, since it has to sample buffer
        n_envs=1,
        env_kwargs={
            "height"  : 8,
            "width"   : 8,
            "obs_type": "data",
    },
        # wrapper_class=RewardShapingWrapper, #TODO: Consider uncommenting it in future runs
        #  wrapper_kwargs={
        #      "step_penalty"   : -0.01,
        #  }
    )
    rewards = []
    for _ in range(n_episodes):
        obs = vec_env_evaluate.reset()
        total = 0
        done = [False]
        while not done[0]:
            #action = model.q_net(torch.tensor(obs, dtype=torch.float32).to(model.device)).argmax(dim=1).cpu().numpy()
            #action = model._get_q(model._discretize(obs)).argmax(axis=1) # because in tabular, values are looked up in the table directly; there is no target nn
            action, _ = model.predict(obs, deterministic=True)
            obs, r, done, _ = vec_env_evaluate.step(action)
            total += r[0]
        rewards.append(total)
    return np.mean(rewards), np.std(rewards)

best_reward = -np.inf

def train_a2c():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {DEVICE}")
    global best_reward
    run = wandb.init(project="Snake_A2C",)
    config = run.config

    # A2C is on-policy — it collects rollouts across all envs before each update.
    # 16 envs means each rollout batch has 16 * n_steps transitions.
    def train_a2c():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {DEVICE}")
    global best_reward
    run = wandb.init(project="Snake_A2C",)
    config = run.config

    # A2C is on-policy — it collects rollouts across all envs before each update.
    # 16 envs means each rollout batch has 16 * n_steps transitions.
    # This mirrors your DQN setup so comparisons are fair.
    vec_env = make_vec_env(
        "tcs-v2",
        n_envs=16,
        env_kwargs={
            "height": 8,
            "width": 8,
            "obs_type": "data",
        },
        wrapper_class=RewardShapingWrapper,
        wrapper_kwargs={
            "step_penalty": config.step_penalty,
        }
    )

    model = CustomA2C(
        env=vec_env,
        state_dim=vec_env.observation_space.shape[0],
        action_dim=vec_env.action_space.n,
        learning_rate=config.learning_rate,
        gamma=config.gamma,
        n_steps=config.n_steps,
        gae_lambda=config.gae_lambda,
        ent_coef=config.ent_coef,
        vf_coef=config.vf_coef,
        max_grad_norm=config.max_grad_norm,
        device=DEVICE,
    )

    model.learn(total_timesteps=3_000_000)
    mean_reward, std_reward = evaluate(model, n_episodes=50)
    wandb.log({
        "mean_reward": mean_reward,
        "std_reward":  std_reward,
    })

    # save only if best across entire sweep
    if mean_reward > best_reward:
        best_reward = mean_reward
        model_path  = "best_a2c_sweep_model.pt" #TRY NEXT MAYBE? "best_sweep_model.npy"
        torch.save({
            "policy":       model.policy.state_dict(),
            "mean_reward":  best_reward,
            "config":       dict(config),
        }, model_path)
        wandb.save(model_path)

        # Mark this run as the best in the sweep
        run.summary["best_reward"] = best_reward
        run.summary["is_best"] = True
        print(f"New sweep best: {best_reward:.2f} | Run: {run.id}")


    #torch.save(model.policy.state_dict(), "a2c_policy.pt")
    #print("Saved to a2c_policy.pt")

    vec_env.close()
    run.finish()



In [ ]:
import torch as th
import numpy as np
from stable_baselines3.common.env_util import make_vec_env
import torch
import numpy as np
from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def eval_after_reloading(model_being_evaluated):
    # Reconstruct model and load weights
    vec_env = make_vec_env(
        "tcs-v2",
        n_envs=1,
        env_kwargs={
            "height": 8, "width": 8,
            "obs_type": "data",
            "render_mode": "human",
            "mp4": True,
        },wrapper_class=RewardShapingWrapper, #TODO: Consider uncommenting it in future runs
        wrapper_kwargs={
            "step_penalty": -0.01,
        }
    )

    #model = A2C(policy="CustomA2CPolicy", env=vec_env, device=DEVICE)
    model = CustomA2C(env=vec_env, device=DEVICE)
    checkpoint = torch.load("best_a2c_sweep_model.pt", weights_only=False)
    model.policy.load_state_dict(checkpoint["policy"])
    model.policy.eval()
    print(f"Best reward: {checkpoint['mean_reward']:.2f} | Best config: {checkpoint['config']}")

    mean_reward, std_reward = evaluate_policy(
        vec_env,
        n_eval_episodes=100,
        deterministic=True,
        render=False,
        warn=True,
    )
    print(f"\nA2C Reward per episode (not mean across episodes): {mean_reward:.2f} | Std: {std_reward:.2f}")
    vec_env.close()


#if __name__ == "__main__":
    #eval_after_reloading("model_debug")
    #eval_after_reloading("model")

In [ ]:
# Grid search set up with wandb

sweep_config_a2c = {
    "method": "bayes",
    "metric": {
        "name": "mean_reward",
        "goal": "maximize"
    },
    "parameters": {
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-4,
            "max": 1e-2
        },
        "gamma": { #experiment with gamma values
            "values": [0.95, 0.97, 0.99, 0.995]
        },
        "n_steps": { # number of steps per episode
            "values": [5, 16, 32]
        },
        "gae_lambda": {
            "values": [0.9, 0.95, 1.0]
        },
        "ent_coef": {
            "distribution": "log_uniform_values",
            "min": 0.001,
            "max": 0.05
        },
        "step_penalty": {
            "values": [-0.001, -0.01, -0.03]
        },
    }
}


if __name__ == "__main__":
    sweep_id = wandb.sweep(sweep_config_a2c, project="Snake_A2C")
    wandb.agent(sweep_id, function=train_a2c, count=50) #TODO: Change to 50 when running experiments, 1 is for debugging


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\adjd119\_netrc.


Create sweep with ID: b95rw5ta
Sweep URL: https://wandb.ai/danylo-zemskyi-city-university-of-london/Snake_A2C/sweeps/b95rw5ta


wandb: Agent Starting Run: ivjw6k9n with config:
wandb: 	ent_coef: 0.0017137010221093126
wandb: 	gae_lambda: 0.9
wandb: 	gamma: 0.99
wandb: 	learning_rate: 0.00030595712018477076
wandb: 	n_steps: 5
wandb: 	step_penalty: -0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\adjd119\_netrc.


Device: cuda


wandb: Currently logged in as: danylo-zemskyi (danylo-zemskyi-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


c:\Users\adjd119\.conda\envs\DRL_env\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
c:\Users\adjd119\.conda\envs\DRL_env\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


In [ ]:
import wandb
import numpy as np

def analyze_sweep(experiment_to_analyze):
    # Load best sweep result
    api = wandb.Api()
    #sweep = api.sweep("danylo-zemskyi-city-university-of-london/Snake_DRL/<YOUR_A2C_SWEEP_ID>")
    sweep = api.sweep(experiment_to_analyze)
    #best_run = max(sweep.runs, key=lambda r: r.summary.get("best_reward", -np.inf))
    best_run = max(sweep.runs, key=lambda r: r.summary.get("loss/total", -np.inf))
    print(f"Best A2C run: {best_run.id} | Reward: {best_run.summary['loss/total'] } | Config: {best_run.config}")
    best_run.file("best_a2c_sweep_model.pt").download(replace=True)

# if __name__ == "__main__":
#     model_to_evaluate = analyze_sweep("danylo-zemskyi-city-university-of-london/Snake_A2C/90fqaeqb")
#     eval_after_reloading("model_to_evaluate")

In [ ]:
def test_policy(policy, env, n_episodes=5, deterministic=True, render=False):
    
    policy
    device = next(policy.parameters()).device
    episode_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        done = [False] * env.num_envs
        total = np.zeros(env.num_envs)

        while not all(done):
            obs_t = torch.tensor(obs, dtype=torch.float32).to(device)
           
            with torch.no_grad():
                obs_tensor = torch.tensor(np.array(obs), dtype=torch.float32).to(device)
                if deterministic:
                    logits = policy.actor(obs_tensor)
                    action = logits.argmax(dim=-1)
                else:
                    action, _, _, _ = policy.get_action_and_value(obs_tensor)


            obs, rewards, done, _ = env.step(action)
            total += np.array(rewards)

        episode_rewards.append(total.mean())

        if render:
            print(f"Episode {ep + 1:>3} | Reward: {total.mean():.2f}")


    env.close()
    mean_r = np.mean(episode_rewards)
    std_r = np.std(episode_rewards)
    print(f"\nResult over {n_episodes} episodes | Mean: {mean_r:.2f} ± {std_r:.2f}")
    return mean_r, std_r

    

eval_env = make_vec_env(
    "tcs-v2",
    env_kwargs={
        "height": 8,
        "width": 8,
        "obs_type": "data",
        "render_mode": "human",
        "mp4": True,        
    },
    n_envs=1,  
)

In [ ]:
checkpoint = torch.load("best_a2c_sweep_model.pt", weights_only=False)


obs_dim = eval_env.observation_space.shape[0]
n_actions = eval_env.action_space.n

policy = A2CNetwork(obs_dim, n_actions)
policy.load_state_dict(checkpoint["policy"])

mean, std = test_policy(policy, eval_env, n_episodes=5, render=True)

print(f"Mean: {mean} | Std: {std}")

Episode   1 | Reward: 2.00
Episode   2 | Reward: 3.00
Episode   3 | Reward: 3.00
Episode   4 | Reward: 3.00
Episode   5 | Reward: 3.00

Result over 5 episodes | Mean: 2.80 ± 0.40
Mean: 2.8 | Std: 0.39999999999999997
